# Lab 12: CLIP and a graph network

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Dec 2, 11:59 PM ET · **Target time:** about 60 minutes

---

## What this lab is for

The last lab, and the lightest, because it is due the day before
presentations. Two halves, matching the lecture.

1. **CLIP, zero-shot.** Classify images into labels it was never trained on,
   using nothing but sentences.
2. **Prompt wording.** Change the sentence, keep everything else, and measure
   the difference.
3. **Retrieval.** Search a pile of images with a sentence.
4. **One graph layer.** Write it by hand, train it on a citation graph, then
   take the edges away and see what is left.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **Everything is seeded, so your numbers should match a classmate's.** A
  mismatch is a bug, or a cell run out of order. Run All from the top.
- **There is no quiz on this lab.** The last quiz is Quiz 12, on Week 12.
  This lab is here because the ideas are worth having in your hands.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.

## Step 0: Setup

::: {.callout-note}
## A GPU helps but is not required
CLIP is the slow part. Embedding 2,000 small images takes seconds on a Colab
GPU and one to two minutes on a laptop CPU. The graph half is under ten
seconds either way. The first run downloads CLIP, about 600 MB, once.
:::

In [ ]:
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor, logging as tlog

tlog.set_verbosity_error()

SEED = 6600
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_IMAGES = 2000

CORA_URL = ("https://benjaminhoughton.georgetown.domains/deep-learning/"
            "labs/lab-12/cora.npz")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def fetch(url, name, env):
    override = os.environ.get(env)
    if override and Path(override).exists():
        return Path(override)
    dest = Path(name)
    if not dest.exists() or dest.stat().st_size < 1000:
        print(f"downloading {name}")
        urllib.request.urlretrieve(url, dest)
    return dest


print(f"torch {torch.__version__}  device {DEVICE}")

---

# Part 1 · Zero-shot classification

CIFAR-10 is ten everyday classes in tiny 32 by 32 photos. CLIP was never
trained on it. You will classify it anyway, using a sentence per class.

In [ ]:
cifar = load_dataset("uoft-cs/cifar10", split="test").select(range(N_IMAGES))
CLASSES = cifar.features["label"].names
images = cifar["img"]
labels = torch.tensor(cifar["label"])

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"{len(images):,} images, classes: {', '.join(CLASSES)}")

Embed every image **once** and keep the vectors. This is the ingestion step
from Week 10: the expensive part happens before anyone asks a question.

In [ ]:
@torch.no_grad()
def embed_images(imgs, batch=250):
    out = []
    for i in range(0, len(imgs), batch):
        b = proc(images=imgs[i:i + batch], return_tensors="pt").to(DEVICE)
        out.append(F.normalize(clip.get_image_features(**b), dim=-1).cpu())
    return torch.cat(out)


@torch.no_grad()
def embed_text(sentences):
    b = proc(text=sentences, return_tensors="pt", padding=True).to(DEVICE)
    return F.normalize(clip.get_text_features(**b), dim=-1).cpu()


image_vecs = embed_images(images)
print(f"stored {tuple(image_vecs.shape)}: one vector per image, "
      f"unit length {image_vecs.norm(dim=1).mean():.3f}")

## 1a · One sentence per class

Return a list of sentences, one per class, by filling each class name into
`template`. A template looks like `"a photo of a {}"`.

In [ ]:
def make_prompts(classes, template):
    """One sentence per class, in the same order as classes."""
    # TODO: template.format(...) for each class name.
    raise NotImplementedError("make_prompts")


print(make_prompts(CLASSES, "a photo of a {}")[:3])

## 1b · Classify

`image_vecs` is `(N, D)` and `text_vecs` is `(K, D)`, one row per class. Both
are unit length, so the cosine similarity is a matrix product. Return the
index of the best-matching class for every image, a tensor of length `N`.

In [ ]:
def zero_shot(image_vecs, text_vecs):
    """Index of the most similar class sentence, for every image."""
    # TODO: similarity matrix, then the argmax along the class dimension.
    raise NotImplementedError("zero_shot")


text_vecs = embed_text(make_prompts(CLASSES, "a photo of a {}"))
pred = zero_shot(image_vecs, text_vecs)
acc = (pred == labels).float().mean().item()
print(f"zero-shot accuracy: {100 * acc:.1f}%   (guessing: 10.0%)")

In [ ]:
conf = torch.zeros(10, 10, dtype=torch.long)
for t, p in zip(labels, pred):
    conf[t, p] += 1

fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.imshow(conf.numpy(), cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.grid(False)
for i in range(10):
    for j in range(10):
        if conf[i, j]:
            ax.text(j, i, int(conf[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if conf[i, j] > conf.max() / 2 else "black")
ax.set_title("No training on CIFAR-10. Only ten sentences.")
fig.tight_layout()

---

# Part 2 · The wording matters

Nothing to fill in. Same images, same model, same vectors. Only the sentence
template changes.

In [ ]:
templates = ["{}", "a photo of a {}", "a blurry photo of a {}"]
results = {}
for tpl in templates:
    tv = embed_text(make_prompts(CLASSES, tpl))
    results[tpl] = 100 * (zero_shot(image_vecs, tv) == labels).float().mean().item()
    print(f"{tpl!r:28} {results[tpl]:.1f}%")

gap = results["a photo of a {}"] - results["{}"]
print(f"\nwording alone moved accuracy by {gap:+.1f} points")

---

# Part 3 · Retrieval

Search is the same dot product, pointed the other way: one sentence against
every stored image.

## 3a · Top matches

`query_vec` is one unit vector of length `D`. Return the indices of the `k`
images most similar to it, best first. `torch.topk` does this.

In [ ]:
def search(query_vec, image_vecs, k=8):
    """Indices of the k most similar images, most similar first."""
    # TODO: similarity of every image to the query, then topk.
    raise NotImplementedError("search")


def show_search(query, k=8):
    idx = search(embed_text([query])[0], image_vecs, k)
    fig, axes = plt.subplots(1, k, figsize=(1.4 * k, 1.9))
    for ax, i in zip(axes, idx.tolist()):
        ax.imshow(images[i])
        ax.set_title(CLASSES[labels[i]], fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    fig.suptitle(f"\"{query}\"", fontsize=12)
    fig.tight_layout()
    return idx


hits = show_search("a photo of a red car")
print("labels of the top 8:", [CLASSES[labels[i]] for i in hits.tolist()])

Try a few of your own. Notice that "red" is not one of the ten classes, and
the search uses it anyway, because the query is a sentence rather than a label.

In [ ]:
for q in ("a horse in a field", "a boat on the water", "an animal at night"):
    show_search(q)

---

# Part 4 · One graph layer

Cora is a citation graph. Each of 2,708 papers is a node, with a 1,433 word
bag-of-words as its features and one of seven subjects as its label. An edge
means one paper cites the other. Only **140** papers have labels you may train
on.

In [ ]:
cora = np.load(fetch(CORA_URL, "cora.npz", "LAB12_CORA"))
X = torch.tensor(cora["X"], dtype=torch.float32)
Y = torch.tensor(cora["y"])
N = len(Y)

A = torch.zeros(N, N)
A[cora["src"], cora["dst"]] = 1.0
A[cora["dst"], cora["src"]] = 1.0          # citations go both ways for us
A.fill_diagonal_(0)

train, val, test = (torch.tensor(cora[k]) for k in ("train", "val", "test"))
print(f"{N:,} papers, {X.shape[1]:,} features, {int(Y.max()) + 1} subjects")
print(f"{int(A.sum()) // 2:,} citations, mean {A.sum(1).mean():.1f} per paper")
print(f"labeled for training: {int(train.sum())}   "
      f"validation: {int(val.sum())}   test: {int(test.sum())}")

## 4a · The layer from the lecture

The lecture wrote one graph convolution, for every node at once, as

$$X' = X W_1 + A X W_2 + b$$

The first term is the node's own features. The second is the **sum** of its
neighbors' features, because row $i$ of $A X$ adds up the rows of $X$ for
every node joined to $i$. Two weight matrices: one for yourself, one shared by
every neighbor.

`self.own` is $W_1$ with the bias, and `self.nbr` is $W_2$. Write `forward`.

In [ ]:
class GraphLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.own = nn.Linear(d_in, d_out)               # W1 and b
        self.nbr = nn.Linear(d_in, d_out, bias=False)   # W2

    def forward(self, x, A):
        """x is (N, d_in), A is (N, N). Return (N, d_out)."""
        # TODO: own features, plus the summed neighbor features. One line.
        raise NotImplementedError("GraphLayer.forward")


layer = GraphLayer(4, 2)
toy_x = torch.eye(4)
toy_A = torch.tensor([[0., 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]])
print("output shape:", tuple(layer(toy_x, toy_A).shape))
with torch.no_grad():
    same = torch.allclose(layer(toy_x, torch.zeros(4, 4)), layer.own(toy_x))
print(f"with no edges, the layer is just its own linear layer: {same}")

That last line is the whole experiment. Hand the layer an adjacency matrix of
zeros and the neighbor term vanishes, leaving an ordinary linear layer. So the
same network, with the edges removed, **is** an MLP.

## 4b · With and without the edges

Nothing to fill in. Two layers, 16 hidden units, 200 epochs, the same seed.
The only difference between the two runs is the matrix passed as `A`.

In [ ]:
class GraphNet(nn.Module):
    def __init__(self, d_in, d_hidden, n_classes):
        super().__init__()
        self.l1 = GraphLayer(d_in, d_hidden)
        self.l2 = GraphLayer(d_hidden, n_classes)

    def forward(self, x, A):
        h = F.relu(self.l1(x, A))
        h = F.dropout(h, 0.5, self.training)
        return self.l2(h, A)


def run(adj, epochs=200):
    torch.manual_seed(SEED)
    net = GraphNet(X.shape[1], 16, int(Y.max()) + 1).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=0.01, weight_decay=5e-4)
    x, y, a = X.to(DEVICE), Y.to(DEVICE), adj.to(DEVICE)
    for _ in range(epochs):
        net.train(); opt.zero_grad()
        F.cross_entropy(net(x, a)[train], y[train]).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        pred = net(x, a).argmax(1).cpu()
    return (pred[test] == Y[test]).float().mean().item()


with_edges = run(A)
no_edges = run(torch.zeros_like(A))
print(f"with the citation edges:  {100 * with_edges:.1f}% on 1,000 test papers")
print(f"edges removed (an MLP):   {100 * no_edges:.1f}%")
print(f"what the edges were worth: {100 * (with_edges - no_edges):+.1f} points")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(["edges removed\n(an MLP)", "with edges"],
        [100 * no_edges, 100 * with_edges], color=["#7f8c8d", "#1c7a3e"])
ax.axvline(100 / 7, color="#a8281f", ls="--", lw=1.5)
ax.text(100 / 7 + 1, -0.45, "guessing", color="#a8281f", fontsize=10)
ax.set_xlim(0, 100)
ax.set_xlabel("test accuracy, %")
ax.set_title("Same model, same 140 labels. Only the edges differ.")
fig.tight_layout()

---

# Part 5 · What this lab showed

Five questions.

1. **Why CLIP can classify CIFAR-10 at all**, never having been trained on it,
   and what the "classifier" actually is.
2. **Why changing the sentence changes the accuracy** when the images, the
   model, and the stored vectors are identical.
3. **Why search and zero-shot classification are the same computation**,
   pointed in different directions.
4. **What a zero adjacency matrix does** to the graph layer, and why that makes
   the comparison in Part 4 fair.
5. **Where the extra points came from.** The model saw the same 140 labels in
   both runs. What else did it see?

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where
you used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that the confusion matrix, the template comparison, the search grids,
   and the edges bar chart are visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Dec 2, 11:59 PM ET**, the day
   before presentations.

Graded for completion. Good luck on Thursday.